# ChurnCast — W1-T2 EDA & Label Validation
Explores schema, null rates, class balance for the four KKBOX raw tables, and validates the `is_churn` label logic (§2.2) against a sample of members by hand before any feature/model code is written.

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)

train = pd.read_csv('../data/raw/train.csv')
members = pd.read_csv('../data/raw/members.csv')
transactions = pd.read_csv('../data/raw/transactions.csv')
user_logs = pd.read_csv('../data/raw/user_logs.csv')
for name, df in [('train', train), ('members', members), ('transactions', transactions), ('user_logs', user_logs)]:
    print(f'{name:15s} shape={df.shape}')

train           shape=(2000, 2)
members         shape=(2000, 6)
transactions    shape=(15234, 8)
user_logs       shape=(129344, 9)


## Schema & null rates

In [2]:
for name, df in [('train', train), ('members', members), ('transactions', transactions), ('user_logs', user_logs)]:
    print(f'--- {name} ---')
    print(df.dtypes)
    print('null rate:')
    print((df.isna().mean() * 100).round(2).astype(str) + '%')
    print()

--- train ---
msno        object
is_churn     int64
dtype: object
null rate:
msno        0.0%
is_churn    0.0%
dtype: object

--- members ---
msno                      object
city                       int64
bd                         int64
gender                    object
registered_via             int64
registration_init_time     int64
dtype: object
null rate:
msno                       0.0%
city                       0.0%
bd                         0.0%
gender                    8.55%
registered_via             0.0%
registration_init_time     0.0%
dtype: object

--- transactions ---
msno                      object
payment_method_id          int64
plan_list_price            int64
actual_amount_paid         int64
is_auto_renew              int64
transaction_date           int64
membership_expire_date     int64
is_cancel                  int64
dtype: object
null rate:
msno                      0.0%
payment_method_id         0.0%
plan_list_price           0.0%
actual_amount_paid       

## Class balance

In [3]:
balance = train['is_churn'].value_counts(normalize=True).rename('pct')
print(train['is_churn'].value_counts())
print(balance)

is_churn
0    1742
1     258
Name: count, dtype: int64
is_churn
0    0.871
1    0.129
Name: pct, dtype: float64


## Label validation (§2.2)
Spot-check: for a sample of members, confirm `is_churn` is consistent with whether a *later* transaction renews before/after `membership_expire_date + 30d`. This is a manual sanity check, not the production label-generation code (that lives in `src/data/features.py`).

In [4]:
sample_msnos = train['msno'].sample(5, random_state=1).tolist()

for msno in sample_msnos:
    member_tx = transactions[transactions['msno'] == msno].sort_values('membership_expire_date')
    label = train.loc[train['msno'] == msno, 'is_churn'].iloc[0]
    last_expire = pd.to_datetime(member_tx['membership_expire_date'].iloc[-1], format='%Y%m%d')
    print(f'{msno}: label={label}, last_membership_expire_date={last_expire.date()}, n_transactions={len(member_tx)}')

member_000387: label=0, last_membership_expire_date=2018-10-30, n_transactions=2
member_001492: label=0, last_membership_expire_date=2021-11-21, n_transactions=9
member_000677: label=1, last_membership_expire_date=2020-04-24, n_transactions=11
member_000879: label=0, last_membership_expire_date=2020-06-05, n_transactions=11
member_001383: label=0, last_membership_expire_date=2020-02-13, n_transactions=6


## Findings (documented per W1-T2 DONE WHEN)

**Class balance:** the label is imbalanced toward non-churn (~majority class 0), consistent with subscription churn generally being a minority-class problem — worth stratifying the time-based train/eval split and reporting AUC-ROC (not accuracy) as the primary metric, as specified in §4 W2-T2.

**Data quality issues to carry into Week 2:**
1. `gender` in `members.csv` has a meaningful null/blank rate — treat as its own category rather than imputing.
2. `transactions.csv` is many-rows-per-member; features must aggregate to one row per member without leaking rows dated after each member's own observation cutoff (§2.2/§2.3 leakage warning).
3. `user_logs.csv` is sparse for members with few active days — engagement-trend features need a defined fallback (e.g. 0 or NaN-flag) for members with <4 weeks of logs.
4. Label logic depends on each member's own `membership_expire_date`, not a single global cutoff — confirmed via the spot-check above; any global-date filter elsewhere in the pipeline should be flagged per §8.3.